# GSM8K Chain-of-Thought with google/gemma-3-4b-it

Query Gemma 3 4B on 5 problems from the GSM8K math reasoning dataset using chain-of-thought prompting.

In [3]:
# Install dependencies if needed
!pip install transformers datasets torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 51.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 56.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 127.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 158.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 65.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 205.6 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 162.7 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 124.2 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 240.0 MB/s  0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 148.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 172.8 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 174.6 MB

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load 5 examples from GSM8K test split
dataset = load_dataset("openai/gsm8k", "main", split="test")
examples = dataset.select(range(5))

print(f"Loaded {len(examples)} examples")
print("\nFirst question preview:")
print(examples[0]["question"])

ModuleNotFoundError: No module named 'datasets'

In [ ]:
MODEL_ID = "google/gemma-3-4b-it"

print(f"Loading tokenizer and model: {MODEL_ID}")
print("This may take a few minutes on first run...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"Model loaded on: {model.device}")

In [ ]:
COT_SYSTEM_PROMPT = (
    "You are a math tutor. Solve the problem step by step, showing your reasoning clearly. "
    "Think through each step before giving the final answer. "
    "End your response with 'The answer is: <number>'"
)

def build_prompt(question: str) -> str:
    """Build a chat-formatted CoT prompt for Gemma instruct."""
    messages = [
        {"role": "system", "content": COT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def generate_answer(question: str, max_new_tokens: int = 512) -> str:
    """Run inference and return the model's response."""
    prompt = build_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy — deterministic CoT
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

In [ ]:
import re

def extract_gt_answer(answer_str: str) -> str:
    """GSM8K ground-truth answers are after '####'."""
    match = re.search(r"####\s*(.+)", answer_str)
    return match.group(1).strip() if match else answer_str.strip()


results = []

for i, example in enumerate(examples):
    question = example["question"]
    gt_answer = extract_gt_answer(example["answer"])

    print(f"{'='*70}")
    print(f"Question {i+1}: {question}")
    print(f"\nGround truth: {gt_answer}")
    print("\nGenerating model response...")

    response = generate_answer(question)

    print(f"\nModel response:\n{response}")

    results.append({
        "question": question,
        "ground_truth": gt_answer,
        "model_response": response,
    })

print(f"\n{'='*70}")
print("Done!")

In [ ]:
# Summary: check how many answers match
def extract_model_answer(response: str) -> str:
    """Extract the final number from 'The answer is: <number>'."""
    match = re.search(r"[Tt]he answer is[:\s]+([\d,\.]+)", response)
    if match:
        return match.group(1).replace(",", "").strip()
    # Fallback: last number in the response
    numbers = re.findall(r"[\d,]+", response)
    return numbers[-1].replace(",", "") if numbers else ""


print("Results summary:")
print(f"{'Q':>2}  {'Ground Truth':>15}  {'Model Answer':>15}  {'Correct?'}")
print("-" * 50)

correct = 0
for i, r in enumerate(results):
    gt = r["ground_truth"].replace(",", "")
    pred = extract_model_answer(r["model_response"])
    is_correct = gt == pred
    if is_correct:
        correct += 1
    print(f"{i+1:>2}  {gt:>15}  {pred:>15}  {'✓' if is_correct else '✗'}")

print("-" * 50)
print(f"Accuracy: {correct}/{len(results)} = {correct/len(results):.0%}")